In [0]:
from pyspark.sql.functions import *

In [0]:

spark.conf.set(
  "fs.azure.account.key.ecommercestorage77.dfs.core.windows.net",
  storage_key
)

In [0]:
silver_path= "abfss://e-commerce@ecommercestorage77.dfs.core.windows.net/Silver"
gold_path="abfss://e-commerce@ecommercestorage77.dfs.core.windows.net/Gold"

In [0]:
df_silver = spark.readStream \
    .format("delta") \
    .load(silver_path)

In [0]:
#Aggregation: Total sales and total items sold per state per min

df_gold=df_silver.withWatermark("timestamp","1 minute")\
    .groupBy(window("timestamp","1 minute"), "state")\
    .agg(sum("total_amount").alias("total_sales"), sum("quantity").alias("total_items_sold"))\
.select(col("window.start").alias("start"), col("window.end").alias("end"), col("state"), col("total_sales"), col("total_items_sold"))

In [0]:
#write to gold layers
df_gold.writeStream.format("delta")\
.outputMode("append")\
.option("checkpointLocation", gold_path+"gold")\
.start(gold_path)

In [0]:
df_gold=spark.read.format("delta").load(gold_path)
display(df_gold)

In [0]:
# Run the git commands in your local development environment's terminal or command prompt where your repository is cloned.